# Vector Search at Scale: brute force, IVF, PQ and IVF-PQ

Companion notebook to [Vector Search at Scale: The Production Engineer's Guide](https://www.mlwhiz.com/p/vector-search-at-scale-the-missing). Each method is implemented twice: an educational version built from NumPy and scikit-learn's `KMeans`, and the optimised FAISS version.

**Benchmark setup**
- Random 128-dimensional Gaussian vectors at 10K, 100K and 1M, generated once with a fixed seed and shared by every method.
- 100 queries per size. Search time is the average per query.
- Recall@10 is measured against the exact nearest neighbours.
- Memory is the size of the built index, not of the raw vectors.
- IVF methods use `nlist = √n` (the post's rule of thumb) and `nprobe = nlist / 10`, the top of the post's suggested starting range of `nlist/100` to `nlist/10`.

Random Gaussian vectors have no cluster structure, which is close to a worst case for IVF and PQ. Real embeddings are much more clustered, so expect higher recall on them at the same settings.

In [1]:
import time

import faiss
import numpy as np

from sklearn.cluster import KMeans

In [2]:
DIM = 128
SIZES = [10_000, 100_000, 1_000_000]  # 10_000_000 may not fit in a laptop's memory
N_QUERIES = 100
K = 10
SEED = 42

rng = np.random.default_rng(SEED)
DATASETS = {
    n: {
        "database": rng.standard_normal((n, DIM), dtype=np.float32),
        "queries": rng.standard_normal((N_QUERIES, DIM), dtype=np.float32),
    }
    for n in SIZES
}


def nlist_for(n):
    """Number of IVF lists: sqrt(n), the post's rule of thumb."""
    return int(np.sqrt(n))


def nprobe_for(nlist):
    """Number of IVF lists to scan per query: nlist / 10, the top of the post's suggested starting range."""
    return max(1, nlist // 10)

In [3]:
# Exact top-K neighbours for every query, used to measure recall.
# IndexFlatL2 is exact brute-force search, just a fast implementation of it.
for n, data in DATASETS.items():
    exact = faiss.IndexFlatL2(DIM)
    exact.add(data["database"])
    _, data["ground_truth"] = exact.search(data["queries"], K)

In [4]:
def index_memory_bytes(index):
    """Memory held by an index: the bytes of every array it stores."""
    if isinstance(index, np.ndarray):
        return index.nbytes
    if isinstance(index, dict):
        return sum(index_memory_bytes(v) for v in index.values())
    if isinstance(index, list):
        return sum(index_memory_bytes(v) for v in index)
    if isinstance(index, faiss.Index):
        return faiss.serialize_index(index).nbytes
    return 0


def profile(name, build_index, search_index, sizes=SIZES, k=K):
    """Build an index for each dataset size, then time search and measure recall@k."""
    results = []
    print(name)

    for n in sizes:
        data = DATASETS[n]

        start = time.perf_counter()
        index = build_index(data["database"])
        build_seconds = time.perf_counter() - start

        start = time.perf_counter()
        found = [search_index(index, query, k) for query in data["queries"]]
        search_ms = (time.perf_counter() - start) / len(data["queries"]) * 1000

        recall = np.mean([
            len(np.intersect1d(f, truth)) / k
            for f, truth in zip(found, data["ground_truth"])
        ])
        memory_mb = index_memory_bytes(index) / 1024**2
        del index

        results.append({
            "name": name,
            "n": n,
            "build_s": build_seconds,
            "search_ms": search_ms,
            "recall": recall,
            "memory_mb": memory_mb,
        })
        print(
            f"  n={n:>9,}: "
            f"build={build_seconds:7.2f}s  "
            f"search={search_ms:7.2f}ms/query  "
            f"recall@{k}={recall:.3f}  "
            f"memory={memory_mb:7.1f}MB"
        )

    return results

# Brute Force

In [5]:
def brute_force_build_index(database):
    """No preparation needed; return the database as the searchable object."""
    return database


def brute_force_search_index(database, query, k=10):
    """Find k nearest neighbors by computing all L2 distances."""
    distances = np.sum((database - query) ** 2, axis=1)
    return np.argsort(distances)[:k]


brute_force_results = profile(
    "Brute force",
    build_index=brute_force_build_index,
    search_index=brute_force_search_index,
)

Brute force
  n=   10,000: build=   0.00s  search=   0.82ms/query  recall@10=1.000  memory=    4.9MB


  n=  100,000: build=   0.00s  search=   9.30ms/query  recall@10=1.000  memory=   48.8MB


  n=1,000,000: build=   0.00s  search= 106.58ms/query  recall@10=1.000  memory=  488.3MB


# IVF: InVerted File index

Pros:
- Faster search than brute force because it searches only selected clusters/lists.

Cons:
- Longer build/index time than brute force because it must train centroids and assign vectors.
- Approximate search: it can miss true nearest neighbors if their clusters are not probed.

Similar:
- Memory usage is roughly the same as brute force for IVF-Flat, plus 8 bytes per vector for ids and a little for the centroids.

## Educational IVF implementation using K-Means

In [6]:
def kmeans_ivf_build_index(database, nlist, train_size=100_000):
    """
    Build an IVF-style index using k-means.

    nlist = number of clusters / inverted lists.
    """
    training_vectors = database[: min(train_size, len(database))]

    kmeans = KMeans(
        n_clusters=nlist,
        random_state=42,
        n_init=1,
    )
    kmeans.fit(training_vectors)

    centroids = kmeans.cluster_centers_.astype(np.float32)

    # Assign every database vector to its nearest centroid.
    labels = kmeans.predict(database)

    inverted_lists = []

    for list_id in range(nlist):
        indices = np.where(labels == list_id)[0]
        vectors = database[indices]

        inverted_lists.append(
            {
                "indices": indices,
                "vectors": vectors,
            }
        )

    return {
        "centroids": centroids,
        "inverted_lists": inverted_lists,
    }


def kmeans_ivf_search_index(index, query, k=10, nprobe=10):
    """
    Search an IVF-style index.

    nprobe = number of centroid lists to scan.
    """
    centroids = index["centroids"]
    inverted_lists = index["inverted_lists"]

    centroid_distances = np.sum((centroids - query) ** 2, axis=1)
    probe_list_ids = np.argsort(centroid_distances)[:nprobe]

    candidate_indices = []
    candidate_vectors = []

    for list_id in probe_list_ids:
        entry = inverted_lists[list_id]

        if len(entry["indices"]) == 0:
            continue

        candidate_indices.append(entry["indices"])
        candidate_vectors.append(entry["vectors"])

    if not candidate_vectors:
        return np.array([], dtype=np.int64)

    candidate_indices = np.concatenate(candidate_indices)
    candidate_vectors = np.vstack(candidate_vectors)

    distances = np.sum((candidate_vectors - query) ** 2, axis=1)
    nearest = np.argsort(distances)[:k]

    return candidate_indices[nearest]


kmeans_ivf_results = profile(
    "KMeans IVF",
    build_index=lambda database: kmeans_ivf_build_index(
        database,
        nlist=nlist_for(len(database)),
    ),
    search_index=lambda index, query, k: kmeans_ivf_search_index(
        index,
        query,
        k=k,
        nprobe=nprobe_for(len(index["centroids"])),
    ),
)

KMeans IVF
  n=   10,000: build=   0.12s  search=   0.11ms/query  recall@10=0.349  memory=    5.0MB


  n=  100,000: build=   3.88s  search=   0.95ms/query  recall@10=0.388  memory=   49.7MB


  n=1,000,000: build=   9.50s  search=  12.77ms/query  recall@10=0.451  memory=  496.4MB


## Optimised FAISS IVF implementation

In [7]:
def faiss_ivf_build_index(database, nlist, train_size=100_000):
    """Build and populate a FAISS IVF index."""
    d = database.shape[1]
    quantizer = faiss.IndexFlatL2(d)
    index = faiss.IndexIVFFlat(quantizer, d, nlist)

    training_data = database[: min(train_size, len(database))]
    index.train(training_data)
    index.add(database)

    return index


def faiss_ivf_search_index(index, query, k=10, nprobe=10):
    """Search a FAISS IVF index, scanning nprobe lists (FAISS defaults to 1)."""
    index.nprobe = nprobe
    distances, indices = index.search(query.reshape(1, -1), k)
    return indices[0]


faiss_ivf_results = profile(
    "FAISS IVF",
    build_index=lambda database: faiss_ivf_build_index(
        database,
        nlist=nlist_for(len(database)),
    ),
    search_index=lambda index, query, k: faiss_ivf_search_index(
        index,
        query,
        k=k,
        nprobe=nprobe_for(index.nlist),
    ),
)

FAISS IVF
  n=   10,000: build=   0.01s  search=   0.02ms/query  recall@10=0.349  memory=    5.0MB


  n=  100,000: build=   0.21s  search=   0.09ms/query  recall@10=0.360  memory=   49.7MB


  n=1,000,000: build=   1.21s  search=   1.14ms/query  recall@10=0.523  memory=  496.4MB


# Product Quantization (PQ)

Pros:
- Much lower memory usage than brute force or IVF-Flat because vectors are stored as compact codes.
- Can make large-scale search practical when full vectors do not fit comfortably in memory.
- Search can be faster than brute force because distances are computed using small lookup tables.

Cons:
- Longer build/index time than brute force because it must train codebooks and encode vectors.
- Search is approximate: compression loses information, so recall can be lower.
- May be slower than IVF-Flat for some sizes/settings unless combined with an inverted index.

## Educational PQ implementation using K-Means

In [8]:
def train_pq_codebooks(training_vectors, m=8, k=256):
    """
    Train Product Quantization codebooks.

    For each subspace, run k-means to find k representative centroids.
    These centroids form the "vocabulary" for encoding subvectors.
    """
    n, d = training_vectors.shape
    d_sub = d // m
    codebooks = []

    for j in range(m):
        # Extract subvectors for this subspace
        start, end = j * d_sub, (j + 1) * d_sub
        subvectors = training_vectors[:, start:end]

        # Run k-means
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=1)
        kmeans.fit(subvectors)

        codebooks.append(kmeans.cluster_centers_.astype(np.float32))

    return codebooks


def encode_pq(vectors, codebooks, chunk_size=20_000):
    """
    Encode vectors as PQ codes (centroid indices).

    Works through the vectors a chunk at a time: comparing every subvector with
    every centroid at once would need an (n, k, d_sub) array, about 8 GB at n=1M.
    """
    n, d = vectors.shape
    m = len(codebooks)
    d_sub = d // m
    k = codebooks[0].shape[0]

    # Use uint8 if k <= 256, else uint16
    dtype = np.uint8 if k <= 256 else np.uint16
    codes = np.zeros((n, m), dtype=dtype)

    for chunk_start in range(0, n, chunk_size):
        chunk = vectors[chunk_start : chunk_start + chunk_size]

        for j, codebook in enumerate(codebooks):
            subvectors = chunk[:, j * d_sub : (j + 1) * d_sub]

            # Find nearest centroid for each subvector
            distances = np.sum(
                (subvectors[:, np.newaxis] - codebook) ** 2, axis=2
            )
            codes[chunk_start : chunk_start + chunk_size, j] = np.argmin(distances, axis=1)

    return codes


def kmeans_pq_build_index(database, m=16, k=256, train_size=100_000):
    training_vectors = database[: min(train_size, len(database))]
    codebooks = train_pq_codebooks(
        training_vectors,
        m=m,
        k=k,
    )
    codes = encode_pq(database, codebooks)
    return {
        "codebooks": codebooks,
        "codes": codes,
    }


def kmeans_pq_search_index(index, query, k=10):
    codebooks = index["codebooks"]
    codes = index["codes"]

    d = query.shape[0]
    m = len(codebooks)
    d_sub = d // m

    lookup_tables = []

    for j, codebook in enumerate(codebooks):
        start, end = j * d_sub, (j + 1) * d_sub
        query_subvector = query[start:end]

        distances = np.sum((codebook - query_subvector) ** 2, axis=1)
        lookup_tables.append(distances)

    approximate_distances = np.zeros(codes.shape[0], dtype=np.float32)

    for j, table in enumerate(lookup_tables):
        approximate_distances += table[codes[:, j]]

    return np.argsort(approximate_distances)[:k]


kmeans_pq_results = profile(
    "KMeans PQ",
    build_index=lambda database: kmeans_pq_build_index(
        database,
        m=16,
        k=256,
        train_size=100_000,
    ),
    search_index=kmeans_pq_search_index,
)

KMeans PQ


  n=   10,000: build=   2.07s  search=   0.64ms/query  recall@10=0.230  memory=    0.3MB


  n=  100,000: build=  35.58s  search=   7.36ms/query  recall@10=0.159  memory=    1.7MB


  n=1,000,000: build=  80.61s  search=  80.80ms/query  recall@10=0.097  memory=   15.4MB


## Optimised FAISS PQ implementation

In [9]:
def faiss_pq_build_index(database, m=16, nbits=8, train_size=100_000):
    d = database.shape[1]
    index = faiss.IndexPQ(d, m, nbits)

    training_data = database[: min(train_size, len(database))]
    index.train(training_data)
    index.add(database)

    return index


def faiss_pq_search_index(index, query, k=10):
    distances, indices = index.search(query.reshape(1, -1), k)
    return indices[0]


faiss_pq_results = profile(
    "FAISS PQ",
    build_index=lambda database: faiss_pq_build_index(
        database,
        m=16,
        nbits=8,
        train_size=100_000,
    ),
    search_index=faiss_pq_search_index,
)

FAISS PQ


  n=   10,000: build=   0.56s  search=   0.04ms/query  recall@10=0.234  memory=    0.3MB


  n=  100,000: build=   0.73s  search=   0.29ms/query  recall@10=0.156  memory=    1.7MB


  n=1,000,000: build=   2.51s  search=   2.87ms/query  recall@10=0.089  memory=   15.4MB


# IVF-PQ: Inverted File index with Product Quantization

Pros:
- Faster search than brute force because it searches only selected inverted lists.
- Much lower memory usage than brute force or IVF-Flat because vectors are stored as compact PQ codes.
- Scales better to very large datasets than plain IVF-Flat when memory is the limiting factor.
- `nprobe` lets you trade search speed for recall.

Cons:
- Longer build/index time than brute force because it must train IVF centroids, train PQ codebooks, and encode vectors.
- Approximate search: it can miss true nearest neighbors due to both cluster pruning and PQ compression.
- Usually has lower recall than IVF-Flat with the same `nprobe`, because PQ stores compressed representations instead of full vectors.
- More parameters to tune, such as `nlist`, `nprobe`, `m`, and `nbits`.

## Educational IVF-PQ implementation using K-Means

1. Use K-Means to train coarse IVF centroids.
2. Assign each database vector to one centroid/list.
3. Compute residuals: `residual = vector - assigned_centroid`
4. Train PQ codebooks on those residuals.
5. Encode each residual as PQ codes.
6. At search time, probe the nearest IVF lists and compare the query residual to compressed residual codes.

In [10]:
def kmeans_ivfpq_build_index(
    database,
    nlist,
    m=16,
    pq_k=256,
    train_size=100_000,
):
    training_vectors = database[: min(train_size, len(database))]

    coarse_kmeans = KMeans(
        n_clusters=nlist,
        random_state=42,
        n_init=1,
    )
    coarse_kmeans.fit(training_vectors)

    centroids = coarse_kmeans.cluster_centers_.astype(np.float32)

    # Assign database vectors to IVF lists.
    labels = coarse_kmeans.predict(database)

    # Train PQ on residuals: vector - assigned coarse centroid.
    training_labels = coarse_kmeans.predict(training_vectors)
    training_residuals = training_vectors - centroids[training_labels]

    codebooks = train_pq_codebooks(
        training_residuals,
        m=m,
        k=pq_k,
    )

    # Encode all database residuals.
    residuals = database - centroids[labels]
    codes = encode_pq(residuals, codebooks)

    inverted_lists = []

    for list_id in range(nlist):
        indices = np.where(labels == list_id)[0]

        inverted_lists.append(
            {
                "indices": indices,
                "codes": codes[indices],
            }
        )

    return {
        "centroids": centroids,
        "codebooks": codebooks,
        "inverted_lists": inverted_lists,
    }


def kmeans_ivfpq_search(index, query, k=10, nprobe=10):
    centroids = index["centroids"]
    codebooks = index["codebooks"]
    inverted_lists = index["inverted_lists"]

    d = query.shape[0]
    m = len(codebooks)
    d_sub = d // m

    centroid_distances = np.sum((centroids - query) ** 2, axis=1)
    probe_list_ids = np.argsort(centroid_distances)[:nprobe]

    all_indices = []
    all_distances = []

    for list_id in probe_list_ids:
        entry = inverted_lists[list_id]

        if len(entry["indices"]) == 0:
            continue

        query_residual = query - centroids[list_id]
        codes = entry["codes"]

        approximate_distances = np.zeros(len(codes), dtype=np.float32)

        for j, codebook in enumerate(codebooks):
            start, end = j * d_sub, (j + 1) * d_sub
            query_subvector = query_residual[start:end]

            subvector_distances = np.sum(
                (codebook - query_subvector) ** 2,
                axis=1,
            )

            approximate_distances += subvector_distances[codes[:, j]]

        all_indices.append(entry["indices"])
        all_distances.append(approximate_distances)

    if not all_indices:
        return np.array([], dtype=np.int64)

    candidate_indices = np.concatenate(all_indices)
    candidate_distances = np.concatenate(all_distances)

    nearest = np.argsort(candidate_distances)[:k]
    return candidate_indices[nearest]


kmeans_ivfpq_results = profile(
    "KMeans IVF-PQ",
    build_index=lambda database: kmeans_ivfpq_build_index(
        database,
        nlist=nlist_for(len(database)),
        m=16,
        pq_k=256,
        train_size=100_000,
    ),
    search_index=lambda index, query, k: kmeans_ivfpq_search(
        index,
        query,
        k=k,
        nprobe=nprobe_for(len(index["centroids"])),
    ),
)

KMeans IVF-PQ


  n=   10,000: build=   2.18s  search=   0.77ms/query  recall@10=0.163  memory=    0.4MB


  n=  100,000: build=  39.62s  search=   2.93ms/query  recall@10=0.125  memory=    2.6MB


  n=1,000,000: build=  96.30s  search=  15.72ms/query  recall@10=0.098  memory=   23.5MB


## Optimised FAISS IVF-PQ Implementation

In [11]:
def faiss_ivfpq_build_index(
    database,
    nlist,
    m=16,
    nbits=8,
    train_size=100_000,
):
    d = database.shape[1]
    quantizer = faiss.IndexFlatL2(d)
    index = faiss.IndexIVFPQ(quantizer, d, nlist, m, nbits)

    training_data = database[: min(train_size, len(database))]
    index.train(training_data)
    index.add(database)

    return index


def faiss_ivfpq_search(index, query, k=10, nprobe=10):
    index.nprobe = nprobe
    distances, indices = index.search(query.reshape(1, -1), k)
    return indices[0]


faiss_ivfpq_results = profile(
    "FAISS IVF-PQ",
    build_index=lambda database: faiss_ivfpq_build_index(
        database,
        nlist=nlist_for(len(database)),
        m=16,
        nbits=8,
        train_size=100_000,
    ),
    search_index=lambda index, query, k: faiss_ivfpq_search(
        index,
        query,
        k=k,
        nprobe=nprobe_for(index.nlist),
    ),
)

FAISS IVF-PQ


  n=   10,000: build=   0.64s  search=   0.26ms/query  recall@10=0.153  memory=    0.4MB


  n=  100,000: build=   0.95s  search=   0.07ms/query  recall@10=0.118  memory=    2.6MB


  n=1,000,000: build=   3.66s  search=   0.55ms/query  recall@10=0.087  memory=   23.5MB


# Summary

All methods side by side, grouped by dataset size.

In [12]:
all_results = (
    brute_force_results
    + kmeans_ivf_results
    + faiss_ivf_results
    + kmeans_pq_results
    + faiss_pq_results
    + kmeans_ivfpq_results
    + faiss_ivfpq_results
)

header = f"{'method':<15}{'build (s)':>11}{'search (ms)':>13}{'recall@10':>11}{'memory (MB)':>13}"

for n in SIZES:
    print(f"\nn = {n:,}")
    print(header)
    for r in all_results:
        if r["n"] == n:
            print(
                f"{r['name']:<15}"
                f"{r['build_s']:>11.2f}"
                f"{r['search_ms']:>13.2f}"
                f"{r['recall']:>11.3f}"
                f"{r['memory_mb']:>13.1f}"
            )


n = 10,000
method           build (s)  search (ms)  recall@10  memory (MB)
Brute force           0.00         0.82      1.000          4.9
KMeans IVF            0.12         0.11      0.349          5.0
FAISS IVF             0.01         0.02      0.349          5.0
KMeans PQ             2.07         0.64      0.230          0.3
FAISS PQ              0.56         0.04      0.234          0.3
KMeans IVF-PQ         2.18         0.77      0.163          0.4
FAISS IVF-PQ          0.64         0.26      0.153          0.4

n = 100,000
method           build (s)  search (ms)  recall@10  memory (MB)
Brute force           0.00         9.30      1.000         48.8
KMeans IVF            3.88         0.95      0.388         49.7
FAISS IVF             0.21         0.09      0.360         49.7
KMeans PQ            35.58         7.36      0.159          1.7
FAISS PQ              0.73         0.29      0.156          1.7
KMeans IVF-PQ        39.62         2.93      0.125          2.6
FAISS IVF-PQ   